# 05 — No-gate audit and the tagging gold sheet

**Purpose.** Two measured guards on the frozen tagging instrument
(`item1-topic-tagging-1.0`), both on seeded random samples:

1. **No-gate audit.** The Pass-A gate exists only to spare Pass B topics a
   block plainly never touches; its cost is missed topics, and that cost is
   *measured here, not assumed*. On `N_AUDIT` random blocks the confirm
   pass runs over **all 111 topics** (in chunks, since 111 definitions do
   not fit one call), and the gate runs as in production. The audit miss
   rate = confirmed (block, topic) tags whose topic the gate did not flag
   on that block. This number goes in the paper next to the gate.
2. **Gold sheet.** A human-codable CSV of confirmed tags plus negative
   probes, the Stage-3 gold discipline: machine generates candidates,
   the researcher fills a `verified` column in a copy that code never
   overwrites; agreement (kappa) is computed in a later session once the
   sheet is coded.

**Inputs.** 03's `blocks.parquet`, 01's sentences, the frozen codebook.

**Outputs.** `tagging/audit/{gate,confirm}.parquet`, audit CallLog;
`data/raw/06_tagging_gold/tagging_gold_candidates.csv` (regenerated
deterministically; the human-coded copy is
`tagging_gold_verified.csv`, never written by code).

**Gate (reported, not blocking).** Audit miss rate with per-topic
breakdown; direction distribution; gold sheet row count.

## Setup and samples

In [ ]:
import hashlib
import os
import sys
import time
from collections import Counter

sys.path.insert(0, "../lib")

import numpy as np
import pandas as pd
import torch

from paths import DATA_REPO, HF_CACHE, TOPICS_INTERIM, relevance_run_dir
import tagging as T
import summarize as S

os.environ["HF_HOME"] = str(HF_CACHE)

MODEL = os.environ.get("MODEL", "Qwen/Qwen3.5-4B")
N_AUDIT = 60                 # audit blocks; ~9 calls each (1 gate + 8 confirm chunks)
AUDIT_TOPICS_PER_CALL = 14   # 111 topics -> 8 confirm chunks
N_GOLD_POS = 100             # confirmed tags sampled into the gold sheet
N_GOLD_NEG = 50              # negative probes (gated-but-unconfirmed pairs)
SEED = 1089

assert not T.PROMPT_VERSION.endswith("-dev"), "audit runs on the frozen prompt"

ITEM1 = relevance_run_dir("item1")
TAG = relevance_run_dir("tagging")
AUDIT = TAG / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)
GOLD_DIR = DATA_REPO / "raw" / "06_tagging_gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

codebook = pd.read_parquet(TOPICS_INTERIM / "taxonomy" / "codebook_v1.parquet")
LABELS_TEXT, TOPIC_IDS = T.format_topic_labels(codebook)
topic_label = codebook.set_index("topic_id")["label"]
all_topics = sorted(TOPIC_IDS)
chunks = [all_topics[i:i + AUDIT_TOPICS_PER_CALL]
          for i in range(0, len(all_topics), AUDIT_TOPICS_PER_CALL)]

blocks = pd.read_parquet(TAG / "blocks.parquet")
meta = pd.read_parquet(
    DATA_REPO / "raw" / "03_sp500_10k_item1" / "sp500_10k_item1.parquet",
    columns=["accessionNo", "conm", "fyear"])
blocks = blocks.merge(meta, on="accessionNo", validate="m:1")
audit_blocks = blocks.sample(N_AUDIT, random_state=SEED).reset_index(drop=True)
sentences = pd.read_parquet(ITEM1 / "sentences.parquet",
                            columns=["accessionNo", "sent_idx", "sentence"])
sent_by_acc = dict(tuple(
    sentences.loc[sentences["accessionNo"].isin(set(audit_blocks["accessionNo"]))]
    .groupby("accessionNo")))

print(f"prompt: {T.PROMPT_VERSION}   audit blocks: {N_AUDIT}   "
      f"confirm chunks per block: {len(chunks)}")
print(f"expected calls: {N_AUDIT * (1 + len(chunks)):,}")

## Model and helpers

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto",
                                             device_map="cuda")
model.eval()
print(f"loaded {MODEL} in {time.time() - t0:.0f}s")


def seed_for(acc, block_idx, stage):
    torch.manual_seed(int.from_bytes(
        hashlib.sha256(f"{acc}:{block_idx}:{stage}".encode()).digest()[:8],
        "little"))


def block_sentences(row):
    grp = sent_by_acc[row.accessionNo]
    sel = grp.loc[(grp["sent_idx"] >= row.sent_lo)
                  & (grp["sent_idx"] <= row.sent_hi), "sentence"].tolist()
    return sel, row.sent_lo + 1

## Run: gate as in production, confirm over all 111 topics

Gate rows and full-scan confirm rows are written per (block, topic); the
miss analysis is a join, so it can be recomputed without re-running.

In [ ]:
log = S.CallLog(AUDIT / "status.jsonl", "audit", MODEL,
                prompt_version=T.PROMPT_VERSION)
gate_rows, confirm_rows = [], []
errs = Counter()
started = time.time()

for row in audit_blocks.itertuples():
    sents, start = block_sentences(row)

    user = T.render_gate_user(sents, start, LABELS_TEXT)
    seed_for(row.accessionNo, row.block_idx, "gate")
    t0 = time.time()
    out, _ = S._generate(model, tokenizer, T.SYSTEM, user, T.GATE_MAX_NEW_TOKENS)
    log.write("gate", 0, row.Index, len(sents), user, out, tokenizer,
              time.time() - t0)
    topics_hit, e = T.parse_gate_topics(out, TOPIC_IDS)
    errs.update(e)
    for tid in topics_hit:
        gate_rows.append({"accessionNo": row.accessionNo,
                          "block_idx": row.block_idx, "topic_id": tid})

    for ci, chunk in enumerate(chunks):
        entries, cand_ids = T.format_topic_entries(codebook, chunk)
        user = T.render_confirm_user(sents, start, entries)
        seed_for(row.accessionNo, row.block_idx, f"audit:{ci}")
        t0 = time.time()
        out, _ = S._generate(model, tokenizer, T.SYSTEM, user,
                             T.CONFIRM_MAX_NEW_TOKENS)
        log.write("confirm", ci, row.Index, len(sents), user, out, tokenizer,
                  time.time() - t0)
        tags, e = T.parse_topic_tags(out, cand_ids, start, len(sents))
        errs.update(e)
        for t in tags:
            confirm_rows.append({
                "accessionNo": row.accessionNo, "block_idx": row.block_idx,
                "sent_display": t["sentence"], "sent_idx": t["sentence"] - 1,
                "topic_id": t["topic"], "direction": t["direction"]})
    done = row.Index + 1
    if done % 10 == 0:
        print(f"  {done}/{N_AUDIT} blocks  "
              f"({(time.time() - started) / 60:.1f} min)")

gate = pd.DataFrame(gate_rows)
confirm = pd.DataFrame(confirm_rows)
gate.to_parquet(AUDIT / "gate.parquet", index=False)
confirm.to_parquet(AUDIT / "confirm.parquet", index=False)
print(f"\ngate rows {len(gate):,}   confirm rows {len(confirm):,}   "
      f"parse errors {dict(errs) or 'none'}")

## The audit number — what the gate misses

A miss is a (block, topic) with at least one full-scan confirmed tag where
the production gate did not flag that topic on that block. The rate is
reported at the pair level and weighted by tags, with the worst topics
named — if one topic carries the misses, the fix is its label, not the
gate.

In [ ]:
conf_pairs = confirm[["accessionNo", "block_idx", "topic_id"]].drop_duplicates()
gate_pairs = set(map(tuple, gate[["accessionNo", "block_idx", "topic_id"]]
                     .itertuples(index=False)))
conf_pairs["gated"] = [
    (a, b, t) in gate_pairs
    for a, b, t in conf_pairs.itertuples(index=False, name=None)]

n_pairs = len(conf_pairs)
n_miss = int((~conf_pairs["gated"]).sum())
print(f"confirmed (block, topic) pairs : {n_pairs}")
print(f"missed by the gate             : {n_miss} "
      f"({n_miss / max(n_pairs, 1):.1%})")

tagged = confirm.merge(conf_pairs, on=["accessionNo", "block_idx", "topic_id"])
n_tags_missed = int((~tagged["gated"]).sum())
print(f"tag-weighted miss rate         : {n_tags_missed}/{len(tagged)} "
      f"({n_tags_missed / max(len(tagged), 1):.1%})")

if n_miss:
    worst = (conf_pairs.loc[~conf_pairs['gated'], 'topic_id']
             .value_counts().head(10))
    print("\nmost-missed topics:")
    for tid, n in worst.items():
        print(f"  {n:>3}x  {tid} {topic_label[tid]}")

print("\ndirections (full scan):",
      dict(confirm['direction'].value_counts()))

## Gold sheet — candidates for human coding

Positives are sampled confirmed tags; negatives are gated-but-unconfirmed
(sentence, topic) probes — the pairs the model itself considered and
declined, which is where a lenient confirm would hide. The researcher fills
`verified` (y/n) and optionally `direction_ok` in the copy; code never
touches the copy.

In [ ]:
rng = np.random.default_rng(SEED)
sent_lookup = sentences.set_index(["accessionNo", "sent_idx"])["sentence"]

pos = confirm.sample(min(N_GOLD_POS, len(confirm)), random_state=SEED).copy()
pos["kind"] = "tag"

gate_cited = gate.merge(audit_blocks[["accessionNo", "block_idx"]])
neg_pool = []
conf_keys = set(map(tuple, confirm[["accessionNo", "block_idx", "topic_id"]]
                    .itertuples(index=False)))
for g in gate_cited.itertuples():
    if (g.accessionNo, g.block_idx, g.topic_id) not in conf_keys:
        row = audit_blocks.loc[(audit_blocks["accessionNo"] == g.accessionNo)
                               & (audit_blocks["block_idx"] == g.block_idx)].iloc[0]
        mid = int((row["sent_lo"] + row["sent_hi"]) // 2)
        neg_pool.append({"accessionNo": g.accessionNo,
                         "block_idx": g.block_idx, "sent_idx": mid,
                         "sent_display": mid + 1, "topic_id": g.topic_id,
                         "direction": "", "kind": "probe"})
neg = pd.DataFrame(neg_pool)
if len(neg) > N_GOLD_NEG:
    neg = neg.sample(N_GOLD_NEG, random_state=SEED)

gold = pd.concat([pos, neg], ignore_index=True)
gold = gold.merge(meta, on="accessionNo")
gold["sentence"] = [
    sent_lookup.get((a, s), "") for a, s in zip(gold["accessionNo"],
                                                gold["sent_idx"])]
gold["topic_label"] = gold["topic_id"].map(topic_label)
gold["verified"] = ""
gold["direction_ok"] = ""
gold = gold.sample(frac=1, random_state=SEED).reset_index(drop=True)

out_csv = GOLD_DIR / "tagging_gold_candidates.csv"
gold[["conm", "fyear", "accessionNo", "block_idx", "sent_display",
      "sentence", "topic_id", "topic_label", "direction", "kind",
      "verified", "direction_ok"]].to_csv(out_csv, index=False)
print(f"wrote {out_csv}  ({len(gold)} rows: "
      f"{(gold['kind'] == 'tag').sum()} tags + "
      f"{(gold['kind'] == 'probe').sum()} probes)")
print("human-coded copy: tagging_gold_verified.csv (never written by code)")

## Next

The audit rate and the coded gold feed the paper's measurement section.
Production (06) does not wait on the gold coding — tags are cheap to
re-aggregate if the gold read forces a prompt revision, which would bump
the version and re-run.